In [1]:
import duckdb
import pandas as pd
from groq import Groq

df = pd.read_parquet('/Users/tejaharshitamullapudi/Downloads/brfss-clinical-ai/data/processed/brfss_clean.parquet')

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print("ready")


ready


In [6]:
SCHEMA = """
You are a SQL expert. Convert the user's question to a DuckDB SQL query.
The table is called 'df' and has these columns:
- diabetes: 'Yes', 'No', 'Yes - pregnancy', 'Pre-diabetes'
- smoking: 'Yes', 'No'
- heart_attack: 'Yes', 'No'
- health_insurance: 'Yes', 'No'
- exercise: 'Yes', 'No'
- alcohol: 'Yes', 'No'
- general_health: 'Excellent', 'Very good', 'Good', 'Fair', 'Poor'
- sex: 'Male', 'Female'
- _STATE: FIPS state code (number)
- _BMI5: BMI multiplied by 100
- MENTHLTH: number of bad mental health days (0-30)
Note: All refused (9), don't know (7), and missing values have already been removed. The data is clean.

Return ONLY the SQL query, nothing else. No explanation, no markdown.
"""

def ask(question):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SCHEMA},
            {"role": "user", "content": question}
        ]
    )
    sql = response.choices[0].message.content.strip()
    result = duckdb.query(sql).df()
    return sql, result

sql, result = ask("What percentage of smokers have diabetes?")
print(sql)
print(result)

SELECT CAST(SUM(CASE WHEN smoking = 'Yes' AND diabetes IN ('Yes', 'Yes - pregnancy', 'Pre-diabetes') THEN 1 ELSE 0 END) AS REAL) * 100 / SUM(CASE WHEN smoking = 'Yes' THEN 1 ELSE 0 END) FROM df
   ((CAST(sum(CASE  WHEN (((smoking = 'Yes') AND (diabetes IN ('Yes', 'Yes - pregnancy', 'Pre-diabetes')))) THEN (1) ELSE 0 END) AS FLOAT) * 100) / sum(CASE  WHEN ((smoking = 'Yes')) THEN (1) ELSE 0 END))
0                                          20.051062                                                                                                                                                        


In [7]:
sql, result = ask("how many people refused to give answer")
print(sql)
print(result)


SELECT COUNT(*) FROM df WHERE diabetes = '9' OR smoking = '9' OR heart_attack = '9' OR health_insurance = '9' OR exercise = '9' OR alcohol = '9' OR general_health = '9' OR sex = '9' OR _STATE = '9' OR _BMI5 = '9' OR MENTHLTH = '9'
   count_star()
0             0
